In [ ]:
# Full name
NAME = ""
# Institutional email (hm.edu or hmtm.de)
EMAIL = ""

<a href="https://colab.research.google.com/github/aica-wavelab/aica-assignments/blob/main/A3_existing_models/10_4_lora.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Using Large Models

+ **AI in Culture and Arts - Tech Crash Course**
+ **Date:** 21.05.2026
+ **Author:** Dr. Benedikt Zönnchen

In [ ]:
#@title install dependencies to play sound
%%capture
print('installing fluidsynth...')
!apt-get install fluidsynth > /dev/null
!cp /usr/share/sounds/sf2/FluidR3_GM.sf2 ./font.sf2
print('done!')

In [ ]:
#@title install dependencies to show score in music notation
%%capture
print('installing musescore3...')
!apt-get install musescore3 > /dev/null
print('done!')

In [ ]:
#@title Setup: install required Python packages

%pip install music21
%pip install pyfluidsynth

%pip install matplotlib
%pip install seaborn

%pip install pandas
%pip install numpy
%pip install torch

%pip install otter-grader==5.5.0

In [ ]:
#@title Setup: download assignment files (run this cell)
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # download test files
    import requests, os

    folders = ['tests', 'data', 'models']
    link = "https://api.github.com/repos/aica-wavelab/aica-assignments/contents/A3_existing_models"

    def download(entry, dest):
        if entry.get('type') != 'file' or not entry.get('download_url'):
            return
        r = requests.get(entry['download_url'])
        r.raise_for_status()
        with open(dest, 'wb') as out:
            out.write(r.content)

    for folder in folders:
        os.makedirs(folder, exist_ok=True)
        for f in requests.get(f"{link}/{folder}").json():
            download(f, f"{folder}/{f['name']}")

    for f in requests.get(link).json():
        if f['name'].endswith('.py'):
            download(f, f['name'])

    # Initialize Otter
    import otter
    grader = otter.Notebook(colab=True)
else:
    import otter
    grader = otter.Notebook('10_4_lora.ipynb')

## 29 LoRA: Parameter-Efficient Fine-Tuning

### The limits of partial fine-tuning

In the previous notebook *28 Transfer Learning* we adapted our folk-song transformer to Bach chorales by **freezing all layers except the last transformer block and the output head**. This is a pragmatic solution — we update only a percentage of parameters and avoid catastrophic forgetting.

But partial fine-tuning has an inherent tension:
- Frozen layers *cannot* contribute to style adaptation at all, even though they also carry style-relevant information.
- Unfreezing more layers risks overfitting on small datasets.

**LoRA** (Low-Rank Adaptation, [Hu et al. 2021](https://arxiv.org/abs/2106.09685)) offers an elegant alternative: instead of selecting layers to update or freeze, it adds tiny *adapter* matrices alongside *every* targeted layer. The pre-trained weights stay frozen forever; only the small adapters are trained.

### The low-rank hypothesis

LoRA rests on a simple observation: when a model adapts from one task to another, the *change* $\Delta W = W_{\text{fine-tuned}} - W_{\text{pre-trained}}$ tends to have **low intrinsic rank** — it lives in a much smaller subspace than the full weight matrix.

If $\Delta W$ is low-rank we can approximate it as a product of two small matrices:

$$\Delta W \approx B A, \quad B \in \mathbb{R}^{d_{\text{out}} \times r},\quad A \in \mathbb{R}^{r \times d_{\text{in}}}, \quad r \ll d$$

The modified forward pass becomes:

$$h = \underbrace{W_0\, x}_{\text{frozen}} + \underbrace{\frac{\alpha}{r}\, B A\, x}_{\text{trainable adapter}}$$

where $\alpha$ is a scaling constant (usually set equal to $r$, giving scale $= 1$). At initialisation $B = 0$, so the adapter contributes nothing and training begins from a perfect copy of the pre-trained model.

### In this notebook we:
1. Visualise how many parameters LoRA trains compared to other strategies.
2. Implement a `LoRALinear` layer from scratch in PyTorch.
3. Inject LoRA adapters into our folk-song transformer.
4. Fine-tune on Bach chorales and compare results to notebook 10_3.

In [ ]:
import math
import copy
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile, glob
import music21 as m21
from torch.utils.data import TensorDataset, DataLoader, random_split

from encoder import PianoRollEncoder, StringToIntEncoder, TERM_SYMBOL
from files import load_midi_files
from transformer import TransformerDecoder

In [ ]:
sns.set_theme(style="whitegrid")
sns.set_context("talk", font_scale=0.8)
sns.set_palette("viridis")
plt.rcParams["figure.figsize"] = (10, 6)

### 29.1 Loading the Pre-trained Folk-Song Model

We start from the same pre-trained transformer used throughout this notebook series — trained on 1,000 German folk songs with 30,188 parameters total.

In [ ]:
with zipfile.ZipFile('data/deu_folk_songs.zip', 'r') as z:
    z.extractall('data/deu_folk_songs/')

time_step = 0.5
mid_files = glob.glob('data/deu_folk_songs/**/*.mid', recursive=True)
folk_streams = load_midi_files(mid_files, time_step=time_step, transpose_to_major=True, max_files=1000)

piano_roll_encoder = PianoRollEncoder(time_step=time_step)
folk_rolls, _      = piano_roll_encoder.encode_streams(folk_streams)
string_to_int      = StringToIntEncoder(folk_rolls)
vocab_size         = len(string_to_int)
print(f'Vocabulary size: {vocab_size}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if not torch.cuda.is_available():
    device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
sequence_len = 64
N_EMBD, N_HEADS, N_BLOCKS = 32, 4, 2

def make_model(dropout=0.0):
    return TransformerDecoder(
        vocab_size=vocab_size, sequence_len=sequence_len,
        n_embd=N_EMBD, n_heads=N_HEADS, n_blocks=N_BLOCKS,
        dropout=dropout
    ).to(device)

pretrained_model = make_model()
pretrained_model.load_state_dict(
    torch.load('models/transformer_model_1000_120.pt', map_location=device)
)
pretrained_model.eval()
n_total = sum(p.numel() for p in pretrained_model.parameters())
print(f'Pre-trained folk-song model loaded. Total parameters: {n_total:,}')

### 29.2 New Data: Bach Chorales

We reuse the same Bach chorale soprano dataset from notebook 10_3 — short Baroque melodies from the `music21` built-in corpus. These provide a stylistically distinct fine-tuning target.

In [ ]:
from music21 import corpus
from files import transpose

print('Loading Bach chorales from the music21 corpus...')
bach_works = corpus.getComposer('bach')

bach_streams = []
for path in bach_works:
    try:
        score = corpus.parse(path)
        if score.parts:
            soprano = score.parts[0]
            soprano = transpose(soprano)
            bach_streams.append(soprano)
    except Exception:
        pass
    if len(bach_streams) >= 60:
        break

print(f'Loaded {len(bach_streams)} Bach chorale soprano parts.')

In [ ]:
bach_rolls, skipped = piano_roll_encoder.encode_streams(bach_streams)
print(f'Encoded {len(bach_rolls)} Bach melodies ({len(skipped)} skipped due to time-step incompatibility).')
print('First Bach melody (first 20 tokens):', bach_rolls[0][:20])
piano_roll_encoder.decode_stream(bach_rolls[0]).show('midi')

### 29.3 Preparing the Fine-tuning Dataset

Identical setup to notebook 10_3: overlapping windows of length `sequence_len`, with targets shifted by one position.

In [ ]:
def make_dataset(rolls, string_to_int, sequence_len, val_frac=0.2, batch_size=64):
    """Convert a list of piano rolls into train/val DataLoaders."""
    i_term    = string_to_int.encode(TERM_SYMBOL)
    rolls_int = string_to_int.encode_sequences(rolls)
    xs, ys = [], []
    for roll in rolls_int:
        padded = [i_term] * sequence_len + roll + [i_term]
        for i in range(len(padded) - sequence_len):
            xs.append(padded[i : i + sequence_len])
            ys.append(padded[i + 1 : i + sequence_len + 1])
    X       = torch.tensor(xs, dtype=torch.long)
    y       = torch.tensor(ys, dtype=torch.long)
    dataset = TensorDataset(X, y)
    val_size   = int(len(dataset) * val_frac)
    train_size = len(dataset) - val_size
    gen        = torch.Generator().manual_seed(42)
    train_set, val_set = random_split(dataset, [train_size, val_size], generator=gen)
    return (DataLoader(train_set, batch_size=batch_size, shuffle=True),
            DataLoader(val_set,   batch_size=batch_size))

train_loader, val_loader = make_dataset(bach_rolls, string_to_int, sequence_len)
print(f'Fine-tuning set: {len(train_loader.dataset)} train / {len(val_loader.dataset)} val windows')

### 29.4 LoRA: Parameter Savings in Numbers

Before implementing LoRA, let us quantify exactly how many parameters it trains for our specific model.

Our transformer has `N_HEADS = 4` attention heads per block and `N_BLOCKS = 2` blocks. Each head contains three linear projections — `key`, `query`, `value` — mapping from `N_EMBD = 32` to `head_size = N_EMBD // N_HEADS = 8`.

For a LoRA adapter on one such layer ($d_{\text{in}} = 32$, $d_{\text{out}} = 8$, rank $r$):
- Matrix $A \in \mathbb{R}^{r \times 32}$: $32r$ parameters
- Matrix $B \in \mathbb{R}^{8 \times r}$: $8r$ parameters
- **Total per layer**: $40r$ parameters — compare to $32 \times 8 = 256$ for full fine-tuning

There are $3 \times 4 \times 2 = 24$ such layers (key/query/value $\times$ heads $\times$ blocks).

The chart below compares trainable parameters across strategies.

In [ ]:
n_total_params = sum(p.numel() for p in pretrained_model.parameters())

# Partial freeze (last block + layer norm + lm_head), as in notebook 10_3
n_partial = sum(
    p.numel()
    for name, p in pretrained_model.named_parameters()
    if 'blocks.1' in name or 'lm_head' in name or 'ln_f' in name
)

# LoRA targeting key/query/value in all heads
# Per LoRALinear: rank * (d_in + d_out)  with d_in=32, d_out=8
n_kqv_layers = 3 * N_HEADS * N_BLOCKS   # = 24
d_in, d_out  = N_EMBD, N_EMBD // N_HEADS

ranks  = [1, 2, 4, 8, 16]
n_lora = [n_kqv_layers * r * (d_in + d_out) for r in ranks]

labels = ['Full\nfine-tuning', 'Partial\nfreeze\n(10_3)'] + [f'LoRA\nr={r}' for r in ranks]
counts = [n_total_params, n_partial] + n_lora
colors = ['#e74c3c', '#e67e22'] + ['#3498db'] * len(ranks)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(labels, counts, color=colors, edgecolor='white', linewidth=0.5)
ax.bar_label(bars, fmt='{:,.0f}', padding=4, fontsize=9)
ax.axhline(n_total_params, color='grey', linestyle='--', linewidth=0.8,
           label=f'Total model params: {n_total_params:,}')
ax.set_ylabel('Trainable parameters')
ax.set_title('Trainable parameter count: fine-tuning strategies compared')
ax.legend()
plt.tight_layout()
plt.show()

print(f'\nTotal model parameters: {n_total_params:,}')
print(f'Partial freeze (10_3): {n_partial:,}  ({100*n_partial/n_total_params:.1f}%)')
for r, n in zip(ranks, n_lora):
    print(f'LoRA r={r:2d}: {n:5,}  ({100*n/n_total_params:.1f}%)')

### 29.5 Implementing a LoRA Layer

A `LoRALinear` module wraps any `nn.Linear` and adds two small adapter matrices $A$ and $B$. The original weights inside the linear layer are permanently frozen; only $A$ and $B$ participate in training.


**Initialisation details:**

- `lora_A` is initialised with Kaiming uniform (same as `nn.Linear` default) — this provides a non-trivial initial subspace.
- `lora_B` is initialised to **zero** — so at epoch 0 the adapter contributes nothing and the model outputs exactly match the pre-trained checkpoint.
- `scale = alpha / rank`. When `alpha == rank` the scale is 1."

---

🖍 **Exercise 29.1:** Complete the `forward` method of `LoRALinear`.

The method should return the sum of:
1. The frozen linear layer output: `self.linear(x)`.
2. The scaled LoRA delta: `self.scale * (x @ self.lora_A.T @ self.lora_B.T)`.

🗣 **Hint:** `lora_A` has shape `(rank, d_in)`, so `lora_A.T` has shape `(d_in, rank)`. For a batched input `x` of shape `(B, T, d_in)`, the matrix multiplication broadcasts automatically.

---

In [ ]:
class LoRALinear(nn.Module):
    """
    A drop-in replacement for nn.Linear that adds a LoRA adapter.

    Forward pass:
        h = W_0 x  +  (alpha / rank) * B A x

    where W_0 is frozen and only B, A are trained.
    B is initialised to zero  -> adapter starts with zero contribution.
    A is initialised with Kaiming uniform (same as nn.Linear default).
    """

    def __init__(self, linear: nn.Linear, rank: int, alpha: float = None):
        super().__init__()
        d_out, d_in = linear.weight.shape
        self.rank  = rank
        self.scale = (float(alpha) if alpha is not None else float(rank)) / rank

        # Keep the frozen pre-trained layer as a registered submodule
        self.linear = linear
        for param in self.linear.parameters():
            param.requires_grad = False

        # Trainable LoRA matrices
        self.lora_A = nn.Parameter(torch.empty(rank, d_in))
        self.lora_B = nn.Parameter(torch.zeros(d_out, rank))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        ...


# Quick sanity check: with lora_B=0 the output must equal the frozen linear
_lin  = nn.Linear(8, 16, bias=False)
_lora = LoRALinear(_lin, rank=2, alpha=2)
_x    = torch.randn(3, 8)
with torch.no_grad():
    assert torch.allclose(_lora(_x), _lin(_x), atol=1e-5), \
        'Sanity check failed: with lora_B=0 outputs should match.'
print('LoRALinear defined and sanity-checked.')

In [ ]:
grader.check("q291")

### 29.6 Injecting LoRA Into the Transformer

The helper function below replaces the `key`, `query`, and `value` linear layers in *every* attention head with `LoRALinear` wrappers. All other parameters — token and position embeddings, feed-forward layers, layer norms, and the output head — are frozen.

This means LoRA adapters are active **in all transformer blocks simultaneously**, which contrasts with the partial-freeze approach in notebook 10_3 that only updated the last block.

In [ ]:
def inject_lora(model: TransformerDecoder, rank: int, alpha: float = None) -> TransformerDecoder:
    """
    Inject LoRA adapters into the key, query, and value projections
    of every attention head. All other parameters are frozen.

    Parameters
    ----------
    model : TransformerDecoder  — modified in-place
    rank  : int                 — LoRA rank r
    alpha : float | None        — scaling constant (defaults to rank, giving scale=1)

    Returns the modified model.
    """
    # Step 1: freeze all existing parameters
    for param in model.parameters():
        param.requires_grad = False

    # Step 2: replace key / query / value in every Head with LoRALinear
    # The new lora_A and lora_B parameters are created with requires_grad=True by default.
    for block in model.blocks:
        for head in block.sa.heads:
            head.key   = LoRALinear(head.key,   rank=rank, alpha=alpha)
            head.query = LoRALinear(head.query, rank=rank, alpha=alpha)
            head.value = LoRALinear(head.value, rank=rank, alpha=alpha)

    return model

---

🖍 **Exercise 29.2:** The cell below creates a LoRA-adapted copy of the pre-trained model with `RANK = 4`.

1. Run the cell to apply LoRA and count the trainable parameters.
2. Store the number of trainable parameters in `n_trainable_lora`.
3. Does the printed number match your expectation from the bar chart in section 29.4?

---

In [ ]:
RANK = 4

# Create a fresh copy so pretrained_model stays unchanged for later comparison
model_lora = inject_lora(copy.deepcopy(pretrained_model), rank=RANK, alpha=RANK)
model_lora.train()
model_lora = model_lora.to(device)

...

n_total = sum(p.numel() for p in model_lora.parameters())
print(f'Trainable parameters: {n_trainable_lora:,} / {n_total:,} ({100*n_trainable_lora/n_total:.1f}%)')
print()
print('Trainable LoRA parameters by name:')
for name, param in model_lora.named_parameters():
    if param.requires_grad:
        print(f'  {name:60s}  shape={str(list(param.shape)):15s}  numel={param.numel():4d}')

In [ ]:
grader.check("q292")

### 29.7 Fine-Tuning with LoRA

The training loop is identical to notebook 10_3 — only the model has changed. The optimizer receives only the trainable LoRA parameters via `filter(lambda p: p.requires_grad, ...)`.

<!-- BEGIN QUESTION -->

---

🖍 **Exercise 29.3:** Before running fine-tuning, think about the following:

1. In notebook 10_3 we updated **46.6%** of parameters (last block + output head). Here we train only **12.7%** (LoRA rank 4). Yet LoRA adapts *all* transformer blocks. How is that possible?
2. At epoch 0, $B = 0$, so the LoRA adapter outputs zero. What does this mean for the model's predictions at the start of training — are they random or meaningful?
3. Predict: do you expect the final validation loss to be higher or lower than the partial-freeze result from notebook 10_3? Why?

---

*Your answer here.*

_Type your answer here, replacing this text._

<!-- END QUESTION -->



In [ ]:
loss_fn   = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model_lora.parameters()),
    lr=1e-4
)

epochs = 200
train_losses, val_losses = [], []

def run_epoch(loader, model, train=True):
    model.train() if train else model.eval()
    total_loss, total = 0.0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits  = model(xb)
            B, T, C = logits.shape
            loss    = loss_fn(logits.view(B * T, C), yb.view(B * T))
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * B * T
            total      += B * T
    return total_loss / total

In [ ]:
for epoch in range(1, epochs + 1):
    tr = run_epoch(train_loader, model_lora, train=True)
    va = run_epoch(val_loader,   model_lora, train=False)
    train_losses.append(tr)
    val_losses.append(va)
    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:2d}/{epochs}  train loss {tr:.4f}  val loss {va:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.lineplot(x=range(1, len(train_losses) + 1), y=train_losses, label='train', ax=ax)
sns.lineplot(x=range(1, len(val_losses)   + 1), y=val_losses,   label='val',   ax=ax)
ax.set_title(f'LoRA fine-tuning loss (rank={RANK}, key/query/value in all heads)')
ax.set_xlabel('Epoch')
ax.set_ylabel('Cross-entropy loss')
plt.tight_layout()
plt.show()

### 29.8 Before vs. After: Listening Comparison

Let us generate melodies with the *original* pre-trained model and the *LoRA fine-tuned* model and listen for stylistic differences.

In [ ]:
def generate(model, seed, string_to_int, temperature=0.9, max_len=100):
    """Generate a melody with temperature sampling."""
    padded   = [TERM_SYMBOL] * sequence_len + seed
    seed_int = string_to_int.encode_sequence(padded)
    melody   = seed[:]
    model.eval()
    with torch.no_grad():
        while True:
            window = seed_int[-sequence_len:]
            idx    = torch.tensor([window], dtype=torch.long, device=device)
            logits = model(idx)[0, -1, :]
            probs  = torch.softmax(logits / temperature, dim=-1)
            sym_i  = torch.multinomial(probs, 1).item()
            seed_int.append(sym_i)
            sym = string_to_int.decode(sym_i)
            if sym == TERM_SYMBOL:
                break
            melody.append(sym)
            if len(melody) >= max_len:
                break
    return melody

SEED = ['60']

In [ ]:
mel_folk = generate(pretrained_model, SEED, string_to_int)
print('Pre-trained model (folk style):')
piano_roll_encoder.decode_stream(mel_folk).show('midi')

In [ ]:
mel_lora = generate(model_lora, SEED, string_to_int)
print(f'LoRA fine-tuned on Bach chorales (rank={RANK}):')
piano_roll_encoder.decode_stream(mel_lora).show('midi')

<!-- BEGIN QUESTION -->

---

🖍 **Exercise 29.4:** Reflect on the results.

1. Listen to both melodies. Do you hear a stylistic difference? What has changed and what has stayed the same?
2. In notebook 10_3 we updated ~47% of parameters (partial freeze of last block + head). Here we updated ~13% (LoRA rank 4). Which approach adapted the model more *broadly* across all layers, and which gave each updated layer more *capacity* to change?
3. LoRA is especially popular for adapting very large language models (tens of billions of parameters) where even partial fine-tuning is computationally infeasible. Why do the parameter savings matter far more there than for our 30K-parameter model?
4. **Optional experiment:** Re-run this notebook with `RANK = 1` and `RANK = 8`. Does melody quality improve with rank? Is there a point of diminishing returns?

---

*Your answer here.*

_Type your answer here, replacing this text._

<!-- END QUESTION -->

### 29.9 Summary

LoRA is a principled, parameter-efficient alternative to partial fine-tuning. Instead of choosing *which* layers to update, it adds small trainable adapters alongside *all* targeted layers while keeping every pre-trained weight intact.

| Strategy | Trainable params | Layers touched | Forgetting risk |
|---|---|---|---|
| Train from scratch | 100% | All | N/A |
| Full fine-tuning | 100% | All | High |
| Partial freeze (10_3) | ~47% | Last block + head | Low |
| **LoRA r=4 (this notebook)** | **~13%** | **All attention layers** | **Very low** |
| LoRA r=1 | ~3% | All attention layers | Very low |

**Key takeaways:**
- The LoRA delta $\Delta W = BA$ starts at zero $\rightarrow$ training begins from the pre-trained model unchanged.
- Only $A$ and $B$ are updated; $W_0$ is never touched.
- Adapters are tiny files that can be saved and swapped independently of the base model.
- For state-of-the-art LLMs (GPT-4, LLaMA, Mixtral) LoRA is now the dominant fine-tuning strategy because it makes adaptation feasible on consumer hardware.

This concludes the **Using Large Models** notebook series. You have explored:
- **26** — Pre-trained model internals and embeddings
- **27** — Decoding strategies (greedy, temperature, top-k, top-p, constrained)
- **28** — Transfer learning by partial fine-tuning
- **29** — Parameter-efficient adaptation with LoRA

Together these techniques form the practical toolkit of modern AI-assisted creative work.

---

## Bonus: LoRA at Scale — GPT-2 on Shakespeare

With our 30,188-parameter folk-song model and only 43 Bach melodies, any fine-tuning effect is subtle.

In this bonus section we use **GPT-2** — a decoder-only transformer with **~124 million parameters** trained by OpenAI on WebText (2019) — to show what LoRA can do at real scale. GPT-2 generates fluent modern English. After LoRA fine-tuning on Shakespeare's plays, the **same frozen base model** generates archaic Elizabethan prose: "thee", "doth", "hath", dramatic monologue structure, iambic cadence — a style shift that is immediately readable.

Fully fine-tuning 124M parameters is expensive. With LoRA rank 8 we train only **~0.5%** of parameters and still achieve a clear, dramatic style transfer.

> **Recommendation:** Run this section on a **GPU runtime** in Google Colab (`Runtime → Change runtime type → T4 GPU`). CPU training works but is slower.

In [ ]:
#@title Bonus: install extra packages
%pip install -q transformers peft datasets accelerate

### B.1 Loading GPT-2

**GPT-2** (Radford et al., 2019) is a decoder-only transformer trained on WebText — roughly 40 GB of text from quality web pages. It generates fluent, coherent modern English.

We use the **base** variant: 124 M parameters, 12 transformer layers, hidden dimension $d = 768$. Its combined Q/K/V attention projection (`c_attn`) is a natural LoRA target.

**Fine-tuning target:** Shakespeare's complete plays (*tiny_shakespeare*). This corpus uses 16th-century English with archaic vocabulary, verse structure, and dramatic dialogue — maximally distinct from GPT-2's modern-English training data, making the style shift easy to see.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if not torch.cuda.is_available():
    device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = 'gpt2'
print('Loading GPT-2 base (~500 MB on first run)...')
gpt2_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token   # GPT-2 has no dedicated pad token

gpt2_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
gpt2_base.eval()

n_gpt2_total = sum(p.numel() for p in gpt2_base.parameters())
print(f'GPT-2 base:          {n_gpt2_total:,} parameters')
print(f'Our folk-song model: {n_total:,} parameters')
print(f'Scale factor:        {n_gpt2_total // n_total:,}x larger')

### B.2 Parameter Savings at This Scale

GPT-2's attention layers have hidden dimension $d = 768$. Full fine-tuning of all 12 layers' Q, K, and V projections would update $12 \times 3 \times 768^2 \approx 21{,}000{,}000$ parameters — roughly a quarter of the entire model. With LoRA rank 8 this drops to $12 \times 2 \times 8 \times 768 \times 3 \approx 442{,}000$: a **47× reduction**, at just 0.52% of total weights.

Compare this to our small model from section 29.4, where the savings ratio was far less dramatic because $d = 8$ is already tiny. **LoRA's efficiency advantage grows with model size.**

In [ ]:
d_gpt2, n_layers_gpt2 = 768, 12
full_attn_b = n_layers_gpt2 * 3 * d_gpt2 * d_gpt2   # Q, K, V per layer × 12 layers

ranks_b  = [2, 4, 8, 16, 32]
lora_b   = [n_layers_gpt2 * 2 * r * d_gpt2 * 3 for r in ranks_b]   # A+B for Q,K,V
labels_b = ['Full attn\nfine-tuning'] + [f'LoRA\nr={r}' for r in ranks_b]
counts_b = [full_attn_b] + lora_b
pcts_b   = [100 * c / n_gpt2_total for c in counts_b]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, vals, fmt, ylabel, title in [
    (axes[0], counts_b, '{:,.0f}', 'Trainable attention params',  'Absolute parameter count'),
    (axes[1], pcts_b,   '{:.2f}%', '% of total model parameters', 'Fraction of model trained'),
]:
    bars = ax.bar(labels_b, vals,
                  color=['#e74c3c'] + ['#3498db'] * len(ranks_b), edgecolor='white')
    ax.bar_label(bars, fmt=fmt, padding=4, fontsize=8)
    ax.set_ylabel(ylabel)
    ax.set_title(f'GPT-2 base (124M params) — {title}')
plt.tight_layout()
plt.show()

for r, n, pct in zip(ranks_b, lora_b, pcts_b[1:]):
    print(f'LoRA r={r:2d}: {n:8,} params  ({pct:.3f}% of {n_gpt2_total:,} total)')

In [ ]:
def generate_text(model, prompt, max_new_tokens=150, temperature=0.9, top_k=50, seed=None):
    """Generate a text continuation from a prompt using GPT-2."""
    if seed is not None:
        torch.manual_seed(seed)
    inputs = gpt2_tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_k=top_k,
            pad_token_id=gpt2_tokenizer.eos_token_id,
        )
    return gpt2_tokenizer.decode(ids[0], skip_special_tokens=True)

# Neutral prompt that both models can complete, but style will differ dramatically.
PROMPT = 'KING: What wouldst thou have of me?'

print('=== Baseline GPT-2 (modern English, no fine-tuning) ===')
base_text_b = generate_text(gpt2_base, PROMPT, seed=42)
print(base_text_b)
print('\n(Notice: modern web-text style, no Shakespearean vocabulary.)')

### B.3 Applying LoRA with `peft`

The HuggingFace `peft` library adds LoRA to any transformer model with a single call. Under the hood it does exactly what our hand-written `LoRALinear` does: freeze all pre-trained weights, add small $A$ and $B$ matrices alongside each targeted layer, and register only those matrices as trainable.

`target_modules=['c_attn']` selects GPT-2's combined Q/K/V projection (a single linear layer that projects from $d=768$ to $3d=2304$). The library automatically handles the frozen/trainable split.

In [ ]:
import copy as _copy
from peft import get_peft_model, LoraConfig, TaskType

gpt2_lora = get_peft_model(
    _copy.deepcopy(gpt2_base),
    LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=8,
        lora_alpha=16,              # scale = alpha / r = 2
        lora_dropout=0.05,
        target_modules=['c_attn'],  # GPT-2 combined Q/K/V projection
        bias='none',
    ),
)
gpt2_lora.print_trainable_parameters()

### B.4 Fine-Tuning on Shakespeare

We load the `tiny_shakespeare` dataset — the complete works of Shakespeare (~1 MB of text) — and split it into fixed-size token chunks for language-model training. Three epochs is enough for a clear style shift on a GPU. On CPU expect ~10–15 minutes.

In [ ]:
from datasets import load_dataset
from torch.utils.data import Dataset as _Dataset, DataLoader as _DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# 1. Load Shakespeare text
print('Loading tiny_shakespeare...')
_ds  = load_dataset('winglian/tiny-shakespeare', split='train')
shakespeare_text = "\n".join(_ds['text'])
print(f'Corpus: {len(shakespeare_text):,} characters')
print(shakespeare_text[:200])

# 2. Chunk into fixed-size token sequences
CHUNK_SIZE = 128

class _ChunkDataset(_Dataset):
    
    def __init__(self, text, tok, chunk_size):
        ids = tok(text, return_tensors='pt').input_ids[0]
        self.chunks = [ids[i : i + chunk_size] for i in range(0, len(ids) - chunk_size + 1, chunk_size)]
    
    def __len__(self):  
        return len(self.chunks)
    
    def __getitem__(self, i):
        ids = self.chunks[i]
        return {'input_ids': ids, 'labels': ids}

def _collate(batch):
    ids = torch.stack([b['input_ids'] for b in batch])
    return {'input_ids': ids, 'labels': ids}

split = int(0.9 * len(shakespeare_text))
print(split)
train_dl_b = _DataLoader(_ChunkDataset(shakespeare_text[:split], gpt2_tokenizer, CHUNK_SIZE), batch_size=8, shuffle=True, collate_fn=_collate)
val_dl_b = _DataLoader(_ChunkDataset(shakespeare_text[split:], gpt2_tokenizer, CHUNK_SIZE), batch_size=8, shuffle=False, collate_fn=_collate)

print(f'\nTrain: {len(train_dl_b.dataset)} chunks  |  Val: {len(val_dl_b.dataset)} chunks  |  Steps/epoch: {len(train_dl_b)}')

In [ ]:
# 3. Training loop
EPOCHS_B = 3
gpt2_lora = gpt2_lora.to(device)
opt_b     = AdamW(filter(lambda p: p.requires_grad, gpt2_lora.parameters()), lr=3e-4)

# Learning rate scheduler which dynamically adjusts the learning rate over time
sched_b   = CosineAnnealingLR(opt_b, T_max=EPOCHS_B * len(train_dl_b))

tr_losses_b, va_losses_b = [], []
for epoch in range(1, EPOCHS_B + 1):
    gpt2_lora.train()
    ep_loss = 0.0
    for step, batch in enumerate(train_dl_b):
        ids  = batch['input_ids'].to(device)

        # Hugging Face automatically shifts the labels by one token to the right internally.
        loss = gpt2_lora(input_ids=ids, labels=ids).loss
        opt_b.zero_grad()
        loss.backward()

        # Avoid exploding model weights
        torch.nn.utils.clip_grad_norm_(gpt2_lora.parameters(), 1.0)

        # Weight manipulation
        opt_b.step()

        # Learning rate manipulation
        sched_b.step()
        
        ep_loss += loss.item()
        if (step + 1) % 20 == 0:
            print(f'  Epoch {epoch}  step {step+1}/{len(train_dl_b)}  loss={loss.item():.4f}', end='\r')
            
    tr_losses_b.append(ep_loss / len(train_dl_b))
    gpt2_lora.eval()
    with torch.no_grad():
        vl = sum(gpt2_lora(input_ids=b['input_ids'].to(device), labels=b['labels'].to(device)).loss.item() for b in val_dl_b) / len(val_dl_b)
    va_losses_b.append(vl)
    print(f'\nEpoch {epoch}/{EPOCHS_B}  train={tr_losses_b[-1]:.4f}  val={va_losses_b[-1]:.4f}')

In [ ]:
fig_b, ax_b = plt.subplots(figsize=(7, 4))
ax_b.plot(range(1, EPOCHS_B + 1), tr_losses_b, marker='o', label='Train')
ax_b.plot(range(1, EPOCHS_B + 1), va_losses_b, marker='s', label='Val')
ax_b.set_xlabel('Epoch'); ax_b.set_ylabel('Cross-entropy loss')
ax_b.set_title('GPT-2 LoRA fine-tuning on Shakespeare (r=8)')
ax_b.legend(); plt.tight_layout(); plt.show()

### B.5 Comparing Base vs. LoRA Model

Both models receive the **same prompt**. Read the outputs and look for:

- **Vocabulary**: does the LoRA model use archaic words like *thee*, *thou*, *doth*, *hath*, *wouldst*, *thine*?
- **Sentence structure**: short declarative sentences (modern) vs. elaborate rhetorical clauses (Shakespearean)?
- **Register**: does the LoRA model maintain dramatic, poetic tone?

The quantitative metric below counts archaic Elizabethan words as a proxy for how much the style has shifted.

In [ ]:
gpt2_lora.eval()

print('=== Base GPT-2 (no fine-tuning) ===')
base_out = generate_text(gpt2_base, PROMPT, seed=7)
print(base_out)

print('\n' + '─' * 60)
print('=== LoRA fine-tuned GPT-2 (Shakespeare, r=8, 3 epochs) ===')
lora_out = generate_text(gpt2_lora, PROMPT, seed=7)
print(lora_out)

In [ ]:
import re

# Archaic Elizabethan words that appear in Shakespeare but almost never in modern web text.
ARCHAIC = {
    'thee', 'thou', 'thy', 'thine', 'doth', 'hath', 'hast', 'wilt', 'shalt',
    'wouldst', 'shouldst', 'couldst', 'art', 'wherefore', 'hence', 'hither',
    'thither', 'ere', 'prithee', 'forsooth', 'methinks', 'perchance', 'verily',
    'nay', 'yea', 'dost', 'twas', 'tis', 'oft', 'whence',
}

def archaic_score(text):
    """Fraction of unique word types that are archaic Elizabethan words."""
    words = set(re.findall(r'\b[a-z]+\b', text.lower()))
    return len(words & ARCHAIC) / max(len(words), 1)

N_SAMPLES = 10
print(f'Sampling {N_SAMPLES} continuations per model (fixed seeds)...')
base_scores = [archaic_score(generate_text(gpt2_base, PROMPT, seed=i)) for i in range(N_SAMPLES)]
lora_scores = [archaic_score(generate_text(gpt2_lora, PROMPT, seed=i)) for i in range(N_SAMPLES)]

print(f'\nArchaic word ratio (higher = more Shakespearean):')
print(f'  Base GPT-2:  avg={sum(base_scores)/N_SAMPLES:.3f}  per sample: {[f"{s:.3f}" for s in base_scores]}')
print(f'  LoRA GPT-2:  avg={sum(lora_scores)/N_SAMPLES:.3f}  per sample: {[f"{s:.3f}" for s in lora_scores]}')

fig_q, ax_q = plt.subplots(figsize=(6, 4))
ax_q.bar(
    ['Base GPT-2', 'LoRA GPT-2\n(r=8, Shakespeare)'], 
    [sum(base_scores) / N_SAMPLES, sum(lora_scores) / N_SAMPLES], 
    color=['#e67e22', '#3498db'], 
    edgecolor='white', 
    width=0.5)
ax_q.set_ylabel('Fraction of archaic Elizabethan word types')
ax_q.set_title('How Shakespearean is the generated text?')
ax_q.set_ylim(0, max(max(base_scores), max(lora_scores)) * 1.5 + 0.01)
plt.tight_layout()
plt.show()

print('\nA higher score for the LoRA model confirms that fine-tuning shifted\n''the language distribution towards Elizabethan English.')